# Auditoria de Qualidade de Dados (Estendida)

Verificações de integridade, anomalias e completude em diversas tabelas, especialmente Faturamento, Vendas, Estoque e Compras.

In [1]:
import duckdb
import os
import glob
import pandas as pd

data_dir = r"E:\repo\lh_nautical_analise\data\raw"
con = duckdb.connect()
print("Conexão estabelecida com DuckDB")

Conexão estabelecida com DuckDB


## 1. Volume de Dados (Linhas por Arquivo)

In [2]:
files = glob.glob(os.path.join(data_dir, "*.csv"))
records = []
for f in files:
    table_name = os.path.basename(f).replace(".csv", "")
    try:
        count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{f}')").fetchone()[0]
        records.append({'Tabela': table_name, 'Linhas': count})
    except Exception as e:
        records.append({'Tabela': table_name, 'Linhas': -1})

pd.DataFrame(records).sort_values('Linhas', ascending=False).reset_index(drop=True)

,Tabela,Linhas
0,order_items,147320
1,stock_movements,115312
2,payments,53546
3,orders,48998
4,fiscal_invoices,34365
5,purchase_order_items,6059
6,stock_levels,6054
7,goods_receipt_items,4733
8,addresses,3998
9,variant_attribute_values,2018


## 2. Duplicidade em Chaves Primárias (PKs)

In [3]:
tables_to_check = ['orders', 'customers', 'products', 'product_variants', 'locations', 'stock_movements', 'purchase_orders']
dup_records = []

for t in tables_to_check:
    f = os.path.join(data_dir, f"{t}.csv")
    try:
        dup_count = con.execute(f"""
            SELECT COUNT(*) FROM (
                SELECT id, COUNT(*) as cnt 
                FROM read_csv_auto('{f}') 
                GROUP BY id 
                HAVING cnt > 1
            )
        """).fetchone()[0]
        dup_records.append({'Tabela': t, 'Duplicatas (id)': dup_count})
    except Exception as e:
        pass

pd.DataFrame(dup_records)

,Tabela,Duplicatas (id)
0,orders,0
1,customers,0
2,products,0
3,product_variants,0
4,locations,0
5,stock_movements,0
6,purchase_orders,0


## 3. Anomalias em Vendas e Varejo

In [4]:
orders_file = os.path.join(data_dir, "orders.csv")
res_orders = con.execute(f"""
    SELECT 
        MIN(placed_at) as min_date,
        MAX(placed_at) as max_date,
        COUNT(*) FILTER (WHERE total < 0) as negative_totals,
        COUNT(*) FILTER (WHERE discount_amount < 0) as negative_discounts,
        COUNT(*) FILTER (WHERE subtotal < 0) as negative_subtotals,
        COUNT(*) FILTER (WHERE status IS NULL) as null_status
    FROM read_csv_auto('{orders_file}')
""").df()
res_orders

,min_date,max_date,negative_totals,negative_discounts,negative_subtotals,null_status
0,2020-01-01 01:19:28,2026-12-31 23:43:09,0,0,0,0


*(Nota-se leakage temporal, com datas até final de 2026. Necessário truncamento no momento de modelagem (< 2026-08-10))*.

## 4. Anomalias no Catálogo (Custos vs Preços)

In [5]:
variants_file = os.path.join(data_dir, "product_variants.csv")
res_var = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE sale_price < cost_price) as prejuizo_na_venda,
        COUNT(*) FILTER (WHERE sale_price <= 0) as preco_zerado_negativo,
        COUNT(*) FILTER (WHERE weight_kg < 0) as peso_negativo
    FROM read_csv_auto('{variants_file}')
""").df()
res_var

,prejuizo_na_venda,preco_zerado_negativo,peso_negativo
0,0,0,0


## 5. Auditoria Adicional: Estoque e Compras

In [7]:
stock_file = os.path.join(data_dir, "stock_levels.csv")
res_stock = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE quantity_on_hand < 0) as estoque_negativo,
        COUNT(*) as total_registros
    FROM read_csv_auto('{stock_file}')
""").df()
res_stock

,estoque_negativo,total_registros
0,0,6054


In [11]:
po_file = os.path.join(data_dir, "purchase_orders.csv")
res_po = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE total < 0) as po_valor_negativo,
        COUNT(*) FILTER (WHERE expected_delivery_at < created_at) as data_entrega_incoerente
    FROM read_csv_auto('{po_file}')
""").df()
res_po

,po_valor_negativo,data_entrega_incoerente
0,0,0
